# Notebook 2 v2 — Simultaneo dFBA (DC dFBA) con semilla desde celda 46

Este notebook ejecuta la version **v2** del modelo simultaneo MPCC
`pFBA_KKT_flux_Zenteno_vargam_simultaneous_v2.jl`, que incluye el estado
`N_rec` (pool de N reciclado por turnover de proteina) y la variante
balanceada de fases, homologa a la celda 46 del notebook 1.

**Semilla**: se carga desde `out/seed_mpcc_from_cell46.h5` (generada por
la nueva celda 47 del notebook 1). La semilla esta en **unidades escaladas**
(c/cs, v/vs) para ser consistente con el MPCC.

Estructura: una celda por bloque logico. Ejecutar en orden secuencial.


## 1. Activacion del entorno y dependencias


In [1]:
import Pkg

function find_project_root(start::AbstractString)
    d = abspath(start)
    while true
        isfile(joinpath(d, "Project.toml")) && return d
        parent = dirname(d)
        parent == d && return nothing
        d = parent
    end
end

project_root = find_project_root(pwd())
project_root === nothing && error("No se encontro Project.toml desde pwd=$(pwd())")

Pkg.activate(project_root)
Pkg.instantiate()

# Dependencias que pueden faltar en el entorno
for pkg in ["HDF5"]
    if Base.find_package(pkg) === nothing
        println("[env] $(pkg) no encontrado. Instalando...")
        Pkg.add(pkg)
    end
end
Pkg.precompile()

println("[env] Proyecto activo: ", project_root)

  Activating project at `c:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\DC_dFBA_Zenteno`


[env] Proyecto activo: c:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\DC_dFBA_Zenteno


In [2]:
# ── Imports ──
using JuMP
using Ipopt
using LinearAlgebra
using DelimitedFiles
using Printf
using Statistics
using Plots
using Libdl
using SparseArrays

# HDF5 para leer semilla
HDF5_AVAILABLE = false
try
    @eval using HDF5
    global HDF5_AVAILABLE = true
    println("[HDF5] cargado OK")
catch e
    @warn "HDF5 no disponible" e
end

[HDF5] cargado OK


## 2. Rutas de archivos producidos por Notebook 1


In [3]:
# Directorio de salida donde NB1 exporta S, bounds, metadata, semilla
OUT_DIR = get(ENV, "OUT_DIR", joinpath(pwd(), "out"))

S_FILE    = joinpath(OUT_DIR, "S.csv")
LB_FILE   = joinpath(OUT_DIR, "lb.csv")
UB_FILE   = joinpath(OUT_DIR, "ub.csv")
RXN_FILE  = joinpath(OUT_DIR, "rxn_ids.txt")
MET_FILE  = joinpath(OUT_DIR, "met_ids.txt")
META_FILE = joinpath(OUT_DIR, "dfba_vargam_metadata.jl")

for f in [S_FILE, LB_FILE, UB_FILE, RXN_FILE, MET_FILE, META_FILE]
    @assert isfile(f) "Archivo no encontrado: $f"
end
println("[paths] Todos los archivos de NB1 encontrados en: ", OUT_DIR)

[paths] Todos los archivos de NB1 encontrados en: c:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\DC_dFBA_Zenteno\out


## 3. Funciones auxiliares (helpers puros)


In [4]:

# Helpers genéricos (puros y reutilizables)
function _meta_get(meta::Dict{String,Any}, key::String, default)
    haskey(meta, key) ? meta[key] : default
end

function _dict_string_string(x)
    d = Dict{String,String}()
    for (k,v) in pairs(x)
        d[string(k)] = string(v)
    end
    return d
end

function _dict_string_float(x)
    d = Dict{String,Float64}()
    for (k,v) in pairs(x)
        d[string(k)] = Float64(v)
    end
    return d
end

function linear_interp(xq::Vector{Float64}, xp::Vector{Float64}, yp::Vector{Float64})
    isempty(xp) && return zeros(length(xq))
    @assert length(xp) == length(yp)
    yq = similar(xq)
    for i in eachindex(xq)
        x = xq[i]
        if x <= xp[1]
            yq[i] = yp[1]
        elseif x >= xp[end]
            yq[i] = yp[end]
        else
            j = searchsortedlast(xp, x)
            x1, x2 = xp[j], xp[j+1]
            y1, y2 = yp[j], yp[j+1]
            θ = (x - x1) / max(x2 - x1, 1e-12)
            yq[i] = y1 + θ * (y2 - y1)
        end
    end
    return yq
end

function _clip(x, lo, hi)
    return min(max(x, lo), hi)
end

function _finite_diff(values::Vector{Float64}, times::Vector{Float64})
    n = length(values)
    out = zeros(n)
    if n <= 1
        return out
    end
    for k in 1:n
        if k == 1
            dt = max(times[2] - times[1], 1e-9)
            out[k] = (values[2] - values[1]) / dt
        elseif k == n
            dt = max(times[n] - times[n-1], 1e-9)
            out[k] = (values[n] - values[n-1]) / dt
        else
            dt = max(times[k+1] - times[k-1], 1e-9)
            out[k] = (values[k+1] - values[k-1]) / dt
        end
    end
    return out
end

softplus_num(x::Float64, ϵ::Float64) = 0.5 * (x + sqrt(x*x + ϵ*ϵ))
sigmoid_num(x::Float64) = 1.0 / (1.0 + exp(-x))

function _percentiles(x::AbstractVector{<:Real})
    isempty(x) && return Dict("p50"=>0.0, "p90"=>0.0, "p95"=>0.0, "p99"=>0.0)
    xv = sort(Float64.(x))
    q(p) = xv[clamp(ceil(Int, p * length(xv)), 1, length(xv))]
    return Dict("p50"=>q(0.50), "p90"=>q(0.90), "p95"=>q(0.95), "p99"=>q(0.99))
end

function _status(max_res::Real, tol::Real)
    max_res <= tol ? "PASS" : (max_res <= 10tol ? "WARN" : "FAIL")
end

function _family_name(s::Int)
    if s == IDX_X
        return "biomasa"
    elseif s in (IDX_G, IDX_F)
        return "azucares"
    elseif s == IDX_NFREE || s in AA_STATE_IDXS
        return "nitrogeno_y_aa"
    elseif s == IDX_E
        return "etanol"
    elseif s in (IDX_PROT, IDX_CARB)
        return "proteina_carbohidrato"
    elseif s in AROMA_STATE_IDXS
        return "aromas"
    else
        return "otros"
    end
end

function _build_time_maps(hvec::Vector{Float64})
    rad = [0.15505102572168, 0.64494897427832, 1.0]
    t_loc = zeros(length(hvec), 3)
    t0 = 0.0
    for i in eachindex(hvec)
        for j in 1:3
            t_loc[i,j] = t0 + rad[j] * hvec[i]
        end
        t0 += hvec[i]
    end
    return t_loc
end



_build_time_maps (generic function with 1 method)

## 4. Carga de S, bounds, metadata, indices de reacciones y selectores

Esta celda carga la matriz estequiometrica, bounds, metadata, y define
todos los indices de reacciones (obj, glu, fru, eth, O2, ATPM, etc.),
`AA_KEYS`, `AROMA_KEYS`, `AA_UPTAKE_IDXS`, `SELECT_UPTAKE/PRODUCT`,
`PAIRWISE_ESTER_IDXS`, `EA_SOFT_*`, etc. Homologa la celda 7 del
notebook sparsepatch.


In [5]:
S = Float64.(readdlm(S_FILE, ','))
lbraw = readdlm(LB_FILE, ',')
ubraw = readdlm(UB_FILE, ',')
RXN_IDS = readlines(RXN_FILE)
MET_IDS = readlines(MET_FILE)

vlb = lbraw isa AbstractVector ? Float64.(lbraw) : Float64.(lbraw[:, 1])
vub = ubraw isa AbstractVector ? Float64.(ubraw) : Float64.(ubraw[:, 1])

nm = size(S, 1)
nv = size(S, 2)

RXN_INDEX = Dict{String, Int}(rid => i for (i, rid) in enumerate(RXN_IDS))
MET_INDEX = Dict{String, Int}(mid => i for (i, mid) in enumerate(MET_IDS))

META = Dict{String,Any}()
if isfile(META_FILE)
    include(META_FILE)
    if @isdefined DFBA_META
        META = deepcopy(DFBA_META)
    else
        # Fallback: old metadata format uses top-level variables, not Dict.
        # Collect critical GAM profile data into META so downstream code works.
        @isdefined(baseline_time_h)       && (META["baseline_time_h"] = baseline_time_h)
        @isdefined(baseline_gam_mmol_gdw) && (META["baseline_gam_mmol_gdw"] = baseline_gam_mmol_gdw)
        println("[META] DFBA_META not found — fallback: collected globals into META")
    end
end

function _meta_get(meta::Dict{String,Any}, key::String, default)
    haskey(meta, key) ? meta[key] : default
end

function _dict_string_string(x)
    d = Dict{String,String}()
    for (k,v) in pairs(x)
        d[string(k)] = string(v)
    end
    return d
end

function _dict_string_float(x)
    d = Dict{String,Float64}()
    for (k,v) in pairs(x)
        d[string(k)] = Float64(v)
    end
    return d
end

OBJ_ID  = _meta_get(META, "obj_id", "r_2111")
GLU_ID  = _meta_get(META, "glu_id", "r_1714")
FRU_ID  = _meta_get(META, "fru_id", "r_1709")
ETH_ID  = _meta_get(META, "eth_id", "r_1761")
O2_ID   = _meta_get(META, "o2_id",  "r_1992")
ATPM_ID = _meta_get(META, "atpm_id","r_4046")
PROT_RXN_ID = _meta_get(META, "prot_rxn_id", "r_4047")

for rid in [OBJ_ID, GLU_ID, FRU_ID, ETH_ID, O2_ID, ATPM_ID, PROT_RXN_ID]
    @assert haskey(RXN_INDEX, rid) "Falta reacción requerida: $(rid)"
end

obj = RXN_INDEX[OBJ_ID]
glu = RXN_INDEX[GLU_ID]
fru = RXN_INDEX[FRU_ID]
eth = RXN_INDEX[ETH_ID]
o2  = RXN_INDEX[O2_ID]
IDX_ATPM = RXN_INDEX[ATPM_ID]
IDX_PROT_RXN = RXN_INDEX[PROT_RXN_ID]

kinetic_n_default = ["r_1654", "r_1879", "r_1891", "r_1889", "r_1906", "r_1911", "r_1873", "r_1912"]
KINETIC_N_SOURCE_IDS = [string(x) for x in _meta_get(META, "KINETIC_N_SOURCE_IDS", kinetic_n_default)]
for rid in KINETIC_N_SOURCE_IDS
    @assert haskey(RXN_INDEX, rid) "Falta fuente cinética de N: $(rid)"
end
KINETIC_N_SOURCE_IDXS = [RXN_INDEX[rid] for rid in KINETIC_N_SOURCE_IDS]

AA_EXCHANGE_MAP = haskey(META, "aa_exchange_ids") ? _dict_string_string(META["aa_exchange_ids"]) :
    Dict("phe"=>"r_1898", "leu"=>"r_1890", "val"=>"r_1910", "met"=>"r_1893", "tyr"=>"r_1914")
for rid in values(AA_EXCHANGE_MAP)
    @assert haskey(RXN_INDEX, rid) "Falta exchange de AA: $(rid)"
end

AROMA_EXCHANGE_MAP = haskey(META, "aroma_exchange_ids") ? _dict_string_string(META["aroma_exchange_ids"]) :
    Dict("pea"=>"r_1590", "isoamyl"=>"r_1865", "isobutanol"=>"r_1866", "methionol"=>"r_1900", "tyrosol"=>"r_1915")
for rid in values(AROMA_EXCHANGE_MAP)
    @assert haskey(RXN_INDEX, rid) "Falta exchange de aroma/alcohol: $(rid)"
end

AA_KEYS = collect(keys(AA_EXCHANGE_MAP))
AROMA_KEYS = collect(keys(AROMA_EXCHANGE_MAP))
sort!(AA_KEYS)
sort!(AROMA_KEYS)

AA_UPTAKE_IDXS = [RXN_INDEX[AA_EXCHANGE_MAP[k]] for k in AA_KEYS]
AROMA_RXN_IDXS = [RXN_INDEX[AROMA_EXCHANGE_MAP[k]] for k in AROMA_KEYS]

UPTAKE_IDXS = vcat([glu, fru], KINETIC_N_SOURCE_IDXS)
PRODUCT_IDXS = [eth, obj]
n_up = length(UPTAKE_IDXS)
n_prod = length(PRODUCT_IDXS)

IS_GLU = [idx == glu ? 1.0 : 0.0 for idx in UPTAKE_IDXS]
IS_FRU = [idx == fru ? 1.0 : 0.0 for idx in UPTAKE_IDXS]
IS_NIT = [1.0 - IS_GLU[i] - IS_FRU[i] for i in eachindex(UPTAKE_IDXS)]

SELECT_UPTAKE = [Float64(mc == UPTAKE_IDXS[k]) for mc in 1:nv, k in 1:n_up]
SELECT_PRODUCT = [Float64(mc == PRODUCT_IDXS[k]) for mc in 1:nv, k in 1:n_prod]

IS_ETH_prod = [1.0, 0.0]
IS_OBJ_prod = [0.0, 1.0]

N_atoms_map = haskey(META, "n_atoms_map") ? _dict_string_float(META["n_atoms_map"]) : Dict{String,Float64}()
N_frac_map  = haskey(META, "n_frac_map")  ? _dict_string_float(META["n_frac_map"])  : Dict{String,Float64}()

MW_N   = 0.014007
MW_GLU = 0.180156
MW_FRU = 0.180156
MW_ETH = 0.046070
MW_O2  = 0.031998

N_atoms_vec = ones(nv)
N_profile_vec = zeros(nv)
for (rid, val) in pairs(N_atoms_map)
    haskey(RXN_INDEX, rid) && (N_atoms_vec[RXN_INDEX[rid]] = Float64(val))
end
for (rid, val) in pairs(N_frac_map)
    haskey(RXN_INDEX, rid) && (N_profile_vec[RXN_INDEX[rid]] = Float64(val))
end

N_frac = zeros(n_up)
for k in 1:n_up
    idx = UPTAKE_IDXS[k]
    if IS_NIT[k] > 0.5
        N_frac[k] = N_profile_vec[idx] / max(N_atoms_vec[idx] * MW_N, 1e-12)
    end
end

AA_ALPHA_MAP = haskey(META, "aa_alpha") ? _dict_string_float(META["aa_alpha"]) :
    Dict(k => 1.0 / max(length(AA_KEYS), 1) for k in AA_KEYS)
AA_ALPHA_VEC = [get(AA_ALPHA_MAP, k, 0.0) for k in AA_KEYS]

AA_MW_MAP = haskey(META, "aa_mw") ? _dict_string_float(META["aa_mw"]) :
    Dict("phe"=>0.16519, "leu"=>0.13117, "val"=>0.11715, "met"=>0.14921, "tyr"=>0.18119)
AROMA_MW_MAP = haskey(META, "aroma_mw") ? _dict_string_float(META["aroma_mw"]) :
    Dict("pea"=>0.12217, "isoamyl"=>0.08815, "isobutanol"=>0.07412, "methionol"=>0.10619, "tyrosol"=>0.13816)
AROMA_MW_VEC = [get(AROMA_MW_MAP, k, 0.1) for k in AROMA_KEYS]

# Pairwise / ethyl acetate defaults aligned with Notebook 1
pairwise_default = [
    ("r_1862", "r_1865", 0.08),
    ("r_1867", "r_1866", 0.08),
    ("r_2000", "r_1589", 0.08),
]
PAIRWISE_META = haskey(META, "pairwise_constraints") ? META["pairwise_constraints"] : pairwise_default
PAIRWISE_ESTER_IDXS = Int[]
PAIRWISE_ALCOHOL_IDXS = Int[]
PAIRWISE_PHI = Float64[]
for row in PAIRWISE_META
    ester_rid = string(row[1])
    alcohol_rid = string(row[2])
    phi = Float64(row[3])
    if haskey(RXN_INDEX, ester_rid) && haskey(RXN_INDEX, alcohol_rid)
        push!(PAIRWISE_ESTER_IDXS, RXN_INDEX[ester_rid])
        push!(PAIRWISE_ALCOHOL_IDXS, RXN_INDEX[alcohol_rid])
        push!(PAIRWISE_PHI, phi)
    end
end
@assert !isempty(PAIRWISE_ESTER_IDXS) "No se detectaron pares ester↔alcohol."

PAIR_ALCOHOL_SELECT = [Float64(mc == PAIRWISE_ALCOHOL_IDXS[p]) for mc in 1:nv, p in 1:length(PAIRWISE_PHI)]
PAIR_ESTER_SELECT   = [Float64(mc == PAIRWISE_ESTER_IDXS[p])   for mc in 1:nv, p in 1:length(PAIRWISE_PHI)]

EA_SOFT_ESTER_IDX = 0
EA_SOFT_ALCOHOL_IDX = 0
PHI_ETHYL_ACETATE_STATIC = 0.0
if haskey(META, "ethyl_acetate_soft")
    ea = META["ethyl_acetate_soft"]
    ester_rid = string(ea["ester_rid"])
    alcohol_rid = string(ea["alcohol_rid"])
    if haskey(RXN_INDEX, ester_rid) && haskey(RXN_INDEX, alcohol_rid)
        EA_SOFT_ESTER_IDX = RXN_INDEX[ester_rid]
        EA_SOFT_ALCOHOL_IDX = RXN_INDEX[alcohol_rid]
        PHI_ETHYL_ACETATE_STATIC = Float64(ea["phi"])
    end
else
    if haskey(RXN_INDEX, "r_1765") && haskey(RXN_INDEX, ETH_ID)
        EA_SOFT_ESTER_IDX = RXN_INDEX["r_1765"]
        EA_SOFT_ALCOHOL_IDX = RXN_INDEX[ETH_ID]
        PHI_ETHYL_ACETATE_STATIC = 0.005
    end
end

EA_ALCOHOL_SELECT = [Float64(mc == EA_SOFT_ALCOHOL_IDX) for mc in 1:nv]
EA_ESTER_SELECT   = [Float64(mc == EA_SOFT_ESTER_IDX) for mc in 1:nv]
OBJ_SELECT  = [Float64(mc == obj) for mc in 1:nv]
ATPM_SELECT = [Float64(mc == IDX_ATPM) for mc in 1:nv]

println("nm=$(nm), nv=$(nv), nAA=$(length(AA_KEYS)), nAroma=$(length(AROMA_KEYS)), nPair=$(length(PAIRWISE_PHI))")
println("[META] keys loaded: $(length(META)) entries")

nm=2806, nv=4131, nAA=5, nAroma=5, nPair=3
[META] keys loaded: 50 entries


## 5. Parametros cineticos, composicion, GAM, perfil termico

Parametros cineticos (MU0, yields, Monod, etc.), funciones de
temperatura y muerte, compute_full_gam. Homologa celda 8 del sparsepatch.


In [6]:
# -----------------------------
# Parámetros cinéticos / composición
# Homologados a NB1 (paper2010)
# -----------------------------
MU0_nom    = 0.18
YXN_nom    = 19.69
YXG_nom    = 1.60
YXF_nom    = 1.60
YEG_nom    = 0.49
YEF_nom    = 0.49
Kn0_nom    = 0.01
Kg0_nom    = 7.5
Kf0_nom    = 7.5
Kig0_nom   = 55.0
Kie0_nom   = 40.0
Kd0_nom    = 0.00044
betaG0_nom = 0.225
betaF0_nom = 0.225
MRATE_0    = 0.01

MU0 = MU0_nom
YEG = YEG_nom
YEF = YEF_nom
YXN = YXN_nom
R = 8.314
EPS = 1e-9

PROT_CONTENT_0 = Float64(_meta_get(META, "PROT_CONTENT_0", 0.46))
CARB_CONTENT_0 = Float64(_meta_get(META, "CARB_CONTENT_0", 0.37))
RNA_FRAC       = Float64(_meta_get(META, "RNA_FRAC", 0.06))
K_DEATH        = Float64(_meta_get(META, "K_DEATH", 0.005))
TURNOVER_LAMBDA = Float64(_meta_get(META, "TURNOVER_LAMBDA", 0.03))
XA_FRACTION     = Float64(_meta_get(META, "XA_FRACTION", 1.0))
N_AMMONIA_FRACTION = Float64(_meta_get(META, "N_AMMONIA_FRACTION", 0.50))
N_TOTAL_DEPLETION_THRESHOLD = Float64(_meta_get(META, "N_TOTAL_DEPLETION_THRESHOLD", 1e-3))
K_AA_UPTAKE_GROWTH = Float64(_meta_get(META, "K_AA_UPTAKE_GROWTH", 0.08))

ATPM_LB_NO_GROWTH = Float64(_meta_get(META, "ATPM_LB_NO_GROWTH", 0.70))
ATPM_UB_NO_GROWTH = Float64(_meta_get(META, "ATPM_UB_NO_GROWTH", 1000.0))

GAM_BASE = Float64(_meta_get(META, "GAM_BASE", 24.7))
GAM_COEFF_P = Float64(_meta_get(META, "GAM_COEFF_P", 16.965))
GAM_COEFF_R = Float64(_meta_get(META, "GAM_COEFF_R", 1.638))
GAM_COEFF_C = Float64(_meta_get(META, "GAM_COEFF_C", 5.210))
Pbase_global = Float64(_meta_get(META, "Pbase_global", PROT_CONTENT_0))
Cbase_global = Float64(_meta_get(META, "Cbase_global", CARB_CONTENT_0))
Rbase_global = Float64(_meta_get(META, "Rbase_global", RNA_FRAC))

function compute_full_gam(P, Rna, Carb, Pbase, Rbase, Cbase)
    Pfactor = P / max(Pbase, 1e-9)
    Rfactor = Rna / max(Rbase, 1e-9)
    Cfactor = max(0.0, (Cbase + Pbase - P - Rna) / max(Cbase, 1e-9))
    return GAM_BASE + GAM_COEFF_P * Pfactor + GAM_COEFF_R * Rfactor + GAM_COEFF_C * Cfactor
end

GAM_REF = compute_full_gam(PROT_CONTENT_0, RNA_FRAC, CARB_CONTENT_0, Pbase_global, Rbase_global, Cbase_global)

# -----------------------------
# Perfil térmico e inyección
# -----------------------------
T_BASE  = try parse(Float64, get(ENV, "T_CONST", "293.15")) catch; 293.15 end
T_STEPS = [36.0, 96.0]
T_DELTAS = [5.0, 3.0]
T_STEEP = 0.5

function dynamic_temperature(t)
    val = T_BASE
    for i in eachindex(T_STEPS)
        σ = 1.0 / (1.0 + exp(-T_STEEP * (t - T_STEPS[i])))
        val += T_DELTAS[i] * σ
    end
    return val
end

function death_rate_T(E, T_val)
    Td = -0.0001 * E^3 + 0.0049 * E^2 - 0.1279 * E + 315.89
    s = 0.5 * (1.0 + tanh(0.5 * (T_val - Td)))
    base = Kd0_nom * exp(0.0415 * E + (130000.0 * (T_val - 305.65)) / (305.65 * R * T_val))
    return base * s
end

SQRT_2PI = sqrt(2.0 * pi)
function smooth_injection(t, t_shot, dose, width)
    abs(t - t_shot) > 5 * width && return 0.0
    return (dose / (width * SQRT_2PI)) * exp(-0.5 * ((t - t_shot) / width)^2)
end

T_INJ_1 = 0.0; DOSE_1 = 0.0; WIDTH_1 = 5.0
T_INJ_2 = 0.0; DOSE_2 = 0.0; WIDTH_2 = 5.0

5.0

## 6. Constructor de perfil GAM exogeno (build_gam_profiles)


In [7]:
# -----------------------------
# GAM exógena desde Notebook 1 (defer hasta tener malla nfe/th)
# -----------------------------

function _vector_from_meta(meta::Dict{String,Any}, key::String)
    if !haskey(meta, key)
        return Float64[]
    end
    return [Float64(x) for x in meta[key]]
end

baseline_time_h = _vector_from_meta(META, "baseline_time_h")
baseline_gam_mmol_gdw = _vector_from_meta(META, "baseline_gam_mmol_gdw")

function build_gam_profiles(nfe_local::Int, th_local::Float64, gam_ref::Float64)
    h_local = fill(th_local / nfe_local, nfe_local)
    tfe = cumsum(h_local)
    if !isempty(baseline_time_h) && !isempty(baseline_gam_mmol_gdw)
        gam_fe = linear_interp(Float64.(tfe), baseline_time_h, baseline_gam_mmol_gdw)
    else
        gam_fe = fill(gam_ref, nfe_local)
    end
    gam_extra = max.(0.0, gam_fe .- gam_ref)
    return gam_fe, gam_extra
end

println(@sprintf("[GAM] baseline points: time=%d gam=%d", length(baseline_time_h), length(baseline_gam_mmol_gdw)))
println("[GAM] Perfil se evaluará en sección de construcción de modelo (nfe/th canónicos).")

[GAM] baseline points: time=73 gam=73
[GAM] Perfil se evaluará en sección de construcción de modelo (nfe/th canónicos).


## 7. Configuracion HSL (COIN-HSL / ma86)


In [8]:
# Configuracion COIN-HSL identica a sparsepatch
const HSL_BIN_DIR = raw"C:\COIN_HSL\CoinHSL.v2023.11.17.x86_64-w64-mingw32-libgfortran5\bin"
const HSL_DLL_FILE = joinpath(HSL_BIN_DIR, "libcoinhsl.dll")

ENV["IPOPT_HSLLIB"] = HSL_DLL_FILE
ENV["HSL_DLL_PATH"] = HSL_DLL_FILE

path_sep = Sys.iswindows() ? ';' : ':'
path_entries = split(get(ENV, "PATH", ""), path_sep; keepempty=false)
if !(HSL_BIN_DIR in path_entries)
    ENV["PATH"] = HSL_BIN_DIR * path_sep * get(ENV, "PATH", "")
end

if isfile(HSL_DLL_FILE)
    try
        h = Libdl.dlopen(HSL_DLL_FILE); Libdl.dlclose(h)
        println("[HSL] libcoinhsl.dll OK")
    catch err
        @warn "HSL dll no se pudo abrir" err
    end
else
    @warn "HSL_DLL_FILE no existe" HSL_DLL_FILE
end
println("[HSL] hsllib = ", ENV["IPOPT_HSLLIB"])

[HSL] libcoinhsl.dll OK
[HSL] hsllib = C:\COIN_HSL\CoinHSL.v2023.11.17.x86_64-w64-mingw32-libgfortran5\bin\libcoinhsl.dll


## 8. Malla temporal, estados, condicion inicial (c0), escalas y parametros MPCC

Define la discretizacion temporal (nfe, ncp, th), indices de estado
(incluido `IDX_NREC` para el pool de N reciclado v2), condiciones
iniciales (`c0`), factores de escala (`cs`, `vs`), selectores KKT,
perfiles GAM, y todos los parametros IPOPT/MPCC.

**Nota**: la variable `c` del MPCC almacena `c_physical / cs`
(valores escalados). La semilla debe estar en estas mismas unidades.


In [9]:
# ─────────────────────────────────────────────────────────────────────────
# Malla temporal (Radau IIA, 3 puntos de colocacion por elemento finito)
# ─────────────────────────────────────────────────────────────────────────
nfe = try parse(Int, get(ENV, "NFE", "18")) catch; 18 end
ncp = 3
th  = try parse(Float64, get(ENV, "TH", "72.0")) catch; 168.0 end
h   = th / nfe
hm  = fill(h, nfe)
var_h = 0.50

# ─────────────────────────────────────────────────────────────────────────
# Indices de estado (deben coincidir con el .jl v2 y con celda 46 de NB1)
# ─────────────────────────────────────────────────────────────────────────
IDX_X     = 1
IDX_NFREE = 2
IDX_G     = 3
IDX_F     = 4
IDX_E     = 5
IDX_O2    = 6
IDX_PROT  = 7
IDX_CARB  = 8

AA_STATE_IDXS = collect(9:(8 + length(AA_KEYS)))
AROMA_STATE_IDXS = collect((9 + length(AA_KEYS)):(8 + length(AA_KEYS) + length(AROMA_KEYS)))

# Estado adicional v2: pool de N reciclado desde turnover de proteina
IDX_NREC = 8 + length(AA_KEYS) + length(AROMA_KEYS) + 1
nc = IDX_NREC   # total de estados

println("nc = $nc  (8 base + $(length(AA_KEYS)) AA + $(length(AROMA_KEYS)) aroma + 1 N_rec)")

# ─────────────────────────────────────────────────────────────────────────
# Constantes v2 (pool de N reciclado y ATPM en fase crecimiento)
# ─────────────────────────────────────────────────────────────────────────
Y_N_FROM_PROT      = 0.16    # fraccion de N liberado por turnover de proteina
K_NREC_UPTAKE      = 0.05    # tasa de drenaje de N_rec para fuelear uptake AA en turnover
BAL_ATPM_GROWTH_LB = 0.7     # piso ATPM durante fase de crecimiento
BAL_ATPM_GROWTH_UB = 1000.0  # techo ATPM durante fase de crecimiento

# ─────────────────────────────────────────────────────────────────────────
# Condiciones iniciales
# ─────────────────────────────────────────────────────────────────────────
X0 = 0.5
N0_total = 0.14
N0_ammonia = N0_total * N_AMMONIA_FRACTION
N0_from_aa = max(0.0, N0_total - N0_ammonia)
AA0_each = (N0_from_aa / MW_N) / max(length(AA_KEYS), 1)

c0 = zeros(nc)
c0[IDX_X] = X0
c0[IDX_NFREE] = N0_ammonia
c0[IDX_G] = 110.0
c0[IDX_F] = 110.0
c0[IDX_E] = 0.0
c0[IDX_O2] = 0.0
c0[IDX_PROT] = X0 * PROT_CONTENT_0
c0[IDX_CARB] = X0 * CARB_CONTENT_0
for a in eachindex(AA_STATE_IDXS)
    c0[AA_STATE_IDXS[a]] = AA0_each
end
for a in eachindex(AROMA_STATE_IDXS)
    c0[AROMA_STATE_IDXS[a]] = 0.0
end
c0[IDX_NREC] = 0.0  # pool N reciclado inicia vacio

println("c0: ", c0)

# ─────────────────────────────────────────────────────────────────────────
# Escalas de estado (cs) -- normalizacion para mejorar condicionamiento
# ─────────────────────────────────────────────────────────────────────────
cs = ones(nc)
cs[IDX_NFREE] = 0.2
cs[IDX_G] = 100.0
cs[IDX_F] = 100.0
cs[IDX_E] = 10.0
cs[IDX_O2] = 0.01
cs[IDX_PROT] = max(0.1, c0[IDX_PROT])
cs[IDX_CARB] = max(0.1, c0[IDX_CARB])
for s in AA_STATE_IDXS
    cs[s] = max(0.1, c0[s])
end
for s in AROMA_STATE_IDXS
    cs[s] = 0.01
end
cs[IDX_NREC] = 0.1  # escala N reciclado

# ─────────────────────────────────────────────────────────────────────────
# Escalas de flujos (vs) -- reescala flujos con bounds grandes
# ─────────────────────────────────────────────────────────────────────────
FLUX_SCALE_TARGET = 50.0
vs = ones(nv)
for rx in 1:nv
    br = max(abs(vlb[rx]), abs(vub[rx]))
    if br > FLUX_SCALE_TARGET
        vs[rx] = br / FLUX_SCALE_TARGET
    end
end

# ─────────────────────────────────────────────────────────────────────────
# Selectores KKT (matrices/vectores auxiliares para stationarity)
# ─────────────────────────────────────────────────────────────────────────
SELECT_AAUPTAKE = [Float64(mc == AA_UPTAKE_IDXS[a]) for mc in 1:nv, a in eachindex(AA_UPTAKE_IDXS)]
D_GROWTH = zeros(nv); D_GROWTH[obj] = -1.0
D_TURNOVER = zeros(nv); D_TURNOVER[IDX_ATPM] = -1.0
D_AAUP = zeros(nv)
for idx in AA_UPTAKE_IDXS
    D_AAUP[idx] = 1e-3
end

# ─────────────────────────────────────────────────────────────────────────
# Perfil GAM exogeno sobre la malla de elementos finitos
# ─────────────────────────────────────────────────────────────────────────
GAM_FE, GAM_EXTRA_FE = build_gam_profiles(nfe, th, GAM_REF)

# ─────────────────────────────────────────────────────────────────────────
# Registro opcional del paquete HSL_jll licenciado
# ─────────────────────────────────────────────────────────────────────────
HSL_JLL_PATH = strip(get(ENV, "HSL_JLL_PATH", ""))
if !isempty(HSL_JLL_PATH)
    try
        Pkg.develop(path = HSL_JLL_PATH)
        println("HSL_jll registrado desde HSL_JLL_PATH = ", HSL_JLL_PATH)
    catch err
        @warn "No se pudo registrar HSL_jll desde HSL_JLL_PATH." exception = (err, catch_backtrace())
    end
end

# ─────────────────────────────────────────────────────────────────────────
# Parametros IPOPT / MPCC
# ─────────────────────────────────────────────────────────────────────────
IPOPT_LINEAR_SOLVER = "ma86"
IPOPT_PRINT_LEVEL = try parse(Int, get(ENV, "IPOPT_PRINT_LEVEL", "5")) catch; 5 end
IPOPT_TOL = try parse(Float64, get(ENV, "IPOPT_TOL", "1e-4")) catch; 1e-4 end
IPOPT_ACCEPTABLE_TOL = try parse(Float64, get(ENV, "IPOPT_ACCEPTABLE_TOL", "1e-2")) catch; 1e-2 end
IPOPT_ACCEPTABLE_ITER = try parse(Int, get(ENV, "IPOPT_ACCEPTABLE_ITER", "12")) catch; 12 end
IPOPT_MAX_ITER = 500
IPOPT_CONSTR_VIOL_TOL = try parse(Float64, get(ENV, "IPOPT_CONSTR_VIOL_TOL", "1e-5")) catch; 1e-5 end
IPOPT_COMPL_INF_TOL = try parse(Float64, get(ENV, "IPOPT_COMPL_INF_TOL", "1e-4")) catch; 1e-4 end
IPOPT_MUMPS_MEM_PERCENT = try parse(Int, get(ENV, "IPOPT_MUMPS_MEM_PERCENT", "20")) catch; 20 end

# IPOPT no soporta rutas con caracteres no-ASCII (ñ, ó, etc.)
# Usamos un directorio temporal con ruta ASCII para el log.
_ipopt_tmp = joinpath(tempdir(), "ipopt_dfba")
mkpath(_ipopt_tmp)
IPOPT_OUTPUT_FILE = joinpath(_ipopt_tmp, "ipopt_log.txt")
println("[IPOPT] output_file = ", IPOPT_OUTPUT_FILE)

# Pesos de penalizacion complementariedad (l1)
PHI_L = 1.0
PHI_U = 1.0
PHI_UPT = 1.0
PHI_PROD = 1.0
PHI_AA = 1.0
PHI_PAIR = 1.0
PHI_EA = 1.0
PHI_ATPM = 1.0
Q_REG = 1e-8
FLUX_SMOOTH_WEIGHT = 1e-8
SOFTPLUS_V_EPS = 1e-6
PHASE_SMOOTH_EPS = 5e-4

APPLY_PRODUCT_CAPS = lowercase(get(ENV, "APPLY_PRODUCT_CAPS", "false")) in ("1", "true", "yes", "on")
EPS_FLUX = try parse(Float64, get(ENV, "EPS_FLUX", "1e-5")) catch; 1e-5 end

println("IPOPT linear solver = ", IPOPT_LINEAR_SOLVER)
println("nc=$(nc), nfe=$(nfe), ncp=$(ncp), th=$(th), apply_caps=$(APPLY_PRODUCT_CAPS)")
println(@sprintf("GAM_FE range = [%.4f, %.4f] | GAM_EXTRA range = [%.4f, %.4f]",
        minimum(GAM_FE), maximum(GAM_FE), minimum(GAM_EXTRA_FE), maximum(GAM_EXTRA_FE)))

nc = 19  (8 base + 5 AA + 5 aroma + 1 N_rec)
c0: [0.5, 0.07, 110.0, 110.0, 0.0, 0.0, 0.23, 0.185, 0.9995002498750626, 0.9995002498750626, 0.9995002498750626, 0.9995002498750626, 0.9995002498750626, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[IPOPT] output_file = C:\Users\CTORRE~1\AppData\Local\Temp\ipopt_dfba\ipopt_log.txt
IPOPT linear solver = ma86
nc=19, nfe=18, ncp=3, th=72.0, apply_caps=false
GAM_FE range = [46.1625, 61.1551] | GAM_EXTRA range = [0.0000, 9.1749]


## 9. Cargar semilla MPCC desde HDF5 (generada por NB1 celda 47)

**Importante sobre escalas**: la semilla debe estar almacenada en
**unidades escaladas** (c/cs, v/vs) para ser consistente con las
variables del MPCC. Si la semilla viene en unidades fisicas, se
reescala aqui antes de pasarla al solver.


In [10]:
# Lectura de seed_mpcc_from_cell46.h5 producido por NB1.
SEED_FILE = joinpath(OUT_DIR, "seed_mpcc_from_cell46.h5")

function _load_seed_h5(path::String)
    isfile(path) || (@warn "SEED_FILE no existe" path; return nothing)
    if !HDF5_AVAILABLE
        @warn "HDF5.jl no cargado; instala/activa entorno y vuelve a correr celda 1 y 2."
        return nothing
    end
    d = Dict{Symbol,Any}()
    h5open(path, "r") do f
        for k in keys(f)
            d[Symbol(k)] = read(f[k])
        end
    end
    return d
end

function _to_tuple(x)
    x isa Tuple && return x
    return Tuple(x)
end

function _coerce_seed_shape(name::Symbol, arr, expected)
    expected_t = _to_tuple(expected)
    sz = size(arr)

    # Exact match
    if sz == expected_t
        return arr, false
    end

    # 2D transpuesta (h5py <-> Julia)
    if length(expected_t) == 2 && ndims(arr) == 2
        if sz == (expected_t[2], expected_t[1])
            return permutedims(arr, (2, 1)), true
        end
    end

    # 3D con ejes invertidos (ej: (ncp,nfe,nc) -> (nc,nfe,ncp))
    if length(expected_t) == 3 && ndims(arr) == 3
        if sz == (expected_t[3], expected_t[2], expected_t[1])
            return permutedims(arr, (3, 2, 1)), true
        end
    end

    # Vector almacenado como matriz fila/columna
    if length(expected_t) == 1 && ndims(arr) == 2
        if sz == (1, expected_t[1]) || sz == (expected_t[1], 1)
            return vec(arr), true
        end
    end

    return arr, false
end

seed_raw = _load_seed_h5(SEED_FILE)

# Dimensiones esperadas por el modelo Julia (para corregir ejes si vienen de h5py)
expected_shapes = Dict{Symbol,Any}(
    :c => (nc, nfe, ncp),
    :cdot => (nc, nfe, ncp),
    :v => (nv, nfe),
    :hv => (nfe,),
    :lambda_ => (nm, nfe),
    :alpha_L => (nv, nfe),
    :alpha_U => (nv, nfe),
    :alpha_upt => (n_up, nfe),
    :alpha_prod => (n_prod, nfe),
    :alpha_aa => (length(AA_KEYS), nfe),
    :alpha_pair => (length(PAIRWISE_PHI), nfe),
    :alpha_ea => (nfe,),
    :alpha_atpm_floor => (nfe,),
    :FO_L => (nv, nfe),
    :FO_U => (nv, nfe),
    :FO_upt => (n_up, nfe),
    :FO_prod => (n_prod, nfe),
    :FO_aa => (length(AA_KEYS), nfe),
    :FO_pair => (length(PAIRWISE_PHI), nfe),
    :FO_ea => (nfe,),
    :FO_atpm => (nfe,),
)

# ── Normalizar orientacion + reescalar semilla (si corresponde) ──
if seed_raw !== nothing
    seed = Dict{Symbol,Any}()

    for (k, v) in seed_raw
        vv = copy(v)
        if haskey(expected_shapes, k)
            vv2, changed = _coerce_seed_shape(k, vv, expected_shapes[k])
            if changed
                println("[seed] ejes corregidos en $(k): $(size(vv)) -> $(size(vv2))")
            end
            vv = vv2
        end
        seed[k] = vv
    end

    # Escalar estados (c, cdot) si vienen en unidades fisicas
    if haskey(seed, :c) && size(seed[:c]) == (nc, nfe, ncp)
        for s in 1:nc, i in 1:nfe, j in 1:ncp
            seed[:c][s, i, j] /= cs[s]
        end
        println("[seed] c escalado: physical -> scaled (c/cs)")
    end

    if haskey(seed, :cdot) && size(seed[:cdot]) == (nc, nfe, ncp)
        for s in 1:nc, i in 1:nfe, j in 1:ncp
            seed[:cdot][s, i, j] /= cs[s]
        end
        println("[seed] cdot escalado: physical -> scaled (cdot/cs)")
    end

    # Escalar flujos (v)
    if haskey(seed, :v) && ndims(seed[:v]) >= 1
        sv = seed[:v]
        if size(sv, 1) == nv
            if ndims(sv) == 2
                for rx in 1:nv, i in 1:size(sv, 2)
                    sv[rx, i] /= vs[rx]
                end
            elseif ndims(sv) == 1
                for rx in 1:nv
                    sv[rx] /= vs[rx]
                end
            end
            println("[seed] v escalado: physical -> scaled (v/vs)")
        else
            @warn "seed[:v] no coincide con nv; se deja sin escalar" size_v=size(sv) nv
        end
    end

    println("[seed] claves cargadas (post-normalizacion):")
    for (k, v) in seed
        println("  ", rpad(String(k), 20), " shape=", size(v))
    end
else
    seed = nothing
    println("[seed] no disponible - se corre sin warm-start de semilla.")
end

[seed] ejes corregidos en alpha_L: (18, 4131) -> (4131, 18)
[seed] ejes corregidos en c: (3, 18, 19) -> (19, 18, 3)
[seed] ejes corregidos en FO_L: (18, 4131) -> (4131, 18)
[seed] ejes corregidos en FO_prod: (18, 2) -> (2, 18)
[seed] ejes corregidos en alpha_prod: (18, 2) -> (2, 18)
[seed] ejes corregidos en FO_aa: (18, 5) -> (5, 18)
[seed] ejes corregidos en cdot: (3, 18, 19) -> (19, 18, 3)
[seed] ejes corregidos en v: (18, 4131) -> (4131, 18)
[seed] ejes corregidos en alpha_upt: (18, 11) -> (11, 18)
[seed] ejes corregidos en FO_upt: (18, 11) -> (11, 18)
[seed] ejes corregidos en alpha_U: (18, 4131) -> (4131, 18)
[seed] ejes corregidos en lambda_: (18, 2806) -> (2806, 18)
[seed] ejes corregidos en alpha_aa: (18, 5) -> (5, 18)
[seed] ejes corregidos en FO_U: (18, 4131) -> (4131, 18)
[seed] c escalado: physical -> scaled (c/cs)
[seed] cdot escalado: physical -> scaled (cdot/cs)
[seed] v escalado: physical -> scaled (v/vs)
[seed] claves cargadas (post-normalizacion):
  alpha_L           

## 10. Preview de la semilla (inspeccion rapida)


In [11]:
# Preview rapido: head de cada variable de la semilla
if seed !== nothing
    for k in (:c, :cdot, :v, :hv, :lambda_, :alpha_L, :alpha_U,
              :alpha_upt, :alpha_prod, :alpha_aa, :alpha_pair,
              :FO_L, :FO_U, :FO_upt, :FO_prod, :FO_aa, :FO_pair,
              :FO_ea, :FO_atpm, :alpha_ea, :alpha_atpm_floor)
        if haskey(seed, k)
            A = seed[k]
            if length(A) > 0
                println("-- $k  shape=$(size(A))  range=[$(minimum(A)), $(maximum(A))]")
            else
                println("-- $k  shape=$(size(A))  (vacio)")
            end
        end
    end
    # Mostrar claves en seed que NO estan en la lista esperada
    expected = Set([:c, :cdot, :v, :hv, :lambda_, :alpha_L, :alpha_U,
                    :alpha_upt, :alpha_prod, :alpha_aa, :alpha_pair,
                    :FO_L, :FO_U, :FO_upt, :FO_prod, :FO_aa, :FO_pair,
                    :FO_ea, :FO_atpm, :alpha_ea, :alpha_atpm_floor])
    extras = setdiff(Set(keys(seed)), expected)
    if !isempty(extras)
        println("
Claves extra en seed (no esperadas por MPCC): ", extras)
    end
else
    println("Sin semilla cargada.")
end

-- c  shape=(19, 18, 3)  range=[0.0, 899.2137908759804]
-- cdot  shape=(19, 18, 3)  range=[-7.834874482284664, 82.75574235728202]
-- v  shape=(4131, 18)  range=[-2.8458917854244103, 5.068229243710214]
-- hv  shape=(18,)  range=[4.0, 4.0]
-- lambda_  shape=(2806, 18)  range=[-1.5827634958897887, 0.7913817479448944]
-- alpha_L  shape=(4131, 18)  range=[-0.316552699177958, -0.0]
-- alpha_U  shape=(4131, 18)  range=[0.0, 2.215868894245705]
-- alpha_upt  shape=(11, 18)  range=[0.0, 0.0]
-- alpha_prod  shape=(2, 18)  range=[0.0, 0.0]
-- alpha_aa  shape=(5, 18)  range=[0.0, 0.0]
-- alpha_pair  shape=(18, 0)  (vacio)
-- FO_L  shape=(4131, 18)  range=[-0.00316552699177958, 0.0]
-- FO_U  shape=(4131, 18)  range=[-2215.868894245705, 3.986411832150929e-16]
-- FO_upt  shape=(11, 18)  range=[0.0, 0.0]
-- FO_prod  shape=(2, 18)  range=[0.0, 0.0]
-- FO_aa  shape=(5, 18)  range=[0.0, 0.0]
-- FO_pair  shape=(18, 0)  (vacio)
-- FO_ea  shape=(18,)  range=[0.0, 0.0]
-- FO_atpm  shape=(18,)  range=[0.0, 0.0

## 11. Construir y resolver el MPCC v2 con la semilla


In [ ]:
include(joinpath("bin", "pFBA_KKT_flux_Zenteno_vargam_simultaneous_v2.jl"))

@assert @isdefined(c0) "c0 no definido. Ejecutar celda de Malla/estados/c0."

println("Lanzando MPCC v2...")
println("  nc=$nc, nfe=$nfe, ncp=$ncp, th=$th")
println("  seed = ", seed === nothing ? "NO" : "SI ($(length(keys(seed))) claves)")

result = pFBA_KKT_flux_Zenteno_vargam_simultaneous_v2(
    c0;
    eps_flux = EPS_FLUX,
    apply_product_caps = APPLY_PRODUCT_CAPS,
    seed = seed,
)

cStar, vStar, cdStar, lStar, alLStar, alUStar,
    auStar, apStar, aaaStar, aprStar, aeaStar, aatpmStar, hStar, diagStar = result

println("\nResolucion terminada.")
println("Status = ", diagStar.solver)

Lanzando MPCC v2...
  nc=19, nfe=18, ncp=3, th=72.0
  seed = SI (21 claves)
[sparse] nnz(S) = 15561 | density = 0.001342


## 12. Postproceso: tiempos, trayectoria y flujos


In [ ]:
# Reconstruir grilla temporal y trayectoria de estados (en unidades FISICAS)
radau_roots = [0.15505102572168, 0.64494897427832, 1.0]
t_fe = [sum(hStar[1:i-1]) for i in 1:nfe]
t_col = Float64[]
for i in 1:nfe, j in 1:ncp
    push!(t_col, t_fe[i] + radau_roots[j] * hStar[i])
end

println("h*: min=$(minimum(hStar))  max=$(maximum(hStar))  sum=$(sum(hStar))")
println("nfe*ncp puntos temporales = ", length(t_col))

# Trayectoria en unidades fisicas: c_physical = cStar * cs
Y = zeros(nc, length(t_col))
for i in 1:nfe, j in 1:ncp
    for s in 1:nc
        Y[s, (i-1)*ncp + j] = cStar[s, i, j] * cs[s]
    end
end
println("Trayectoria shape: ", size(Y))

## 13. Plots de estados principales


In [ ]:
plot_rows = [
    (IDX_X,     "X (gDW/L)"),
    (IDX_NFREE, "N libre (gN/L)"),
    (IDX_G,     "Glucosa (g/L)"),
    (IDX_F,     "Fructosa (g/L)"),
    (IDX_E,     "Etanol (g/L)"),
    (IDX_PROT,  "Prot (g/L)"),
    (IDX_CARB,  "Carb (g/L)"),
    (IDX_NREC,  "N_rec (gN/L)"),
]

plts = [plot(t_col, Y[s,:], title=ttl, lw=2, xlabel="t [h]", legend=false)
        for (s, ttl) in plot_rows]
plot(plts..., layout=(length(plts), 1), size=(700, 200*length(plts)))

## 14. Exportar resultados a CSV


In [ ]:
# Trayectoria de estados (unidades fisicas)
open(joinpath(OUT_DIR, "xk_simultaneous_v2.csv"), "w") do f
    for i in 1:size(Y,2)
        println(f, join(Y[:,i], ","))
    end
end

# Flujos (unidades fisicas: v_physical = vStar * vs)
open(joinpath(OUT_DIR, "v_simultaneous_v2.csv"), "w") do f
    for i in 1:nfe
        v_phys = [vStar[rx,i] * vs[rx] for rx in 1:nv]
        println(f, join(v_phys, ","))
    end
end

# Tiempos
open(joinpath(OUT_DIR, "t_simultaneous_v2.csv"), "w") do f
    for t in t_col; println(f, t); end
end

println("CSV v2 exportados a ", OUT_DIR)